In [1]:
import os
import fitz  # PyMuPDF
import chromadb
from sentence_transformers import SentenceTransformer
import ollama

# Set up paths
PDF_DIR = "path/to/your/pdf/folder"

# Load Sentence Transformer model
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

# Initialize ChromaDB client
chroma_client = chromadb.PersistentClient(path="./chroma_db")
collection = chroma_client.get_or_create_collection(name="pdf_vectors")


def extract_text_from_pdf(pdf_path):
    """Extract text from a PDF file."""
    doc = fitz.open(pdf_path)
    text = ""
    for page in doc:
        text += page.get_text()
    return text


def index_pdfs(pdf_dir):
    """Convert PDFs to vector embeddings and store in ChromaDB."""
    for root, _, files in os.walk(pdf_dir):
        for file in files:
            if file.endswith(".pdf"):
                file_path = os.path.join(root, file)
                text = extract_text_from_pdf(file_path)
                if text.strip():
                    embedding = embedding_model.encode(text).tolist()
                    collection.add(
                        ids=[file],
                        embeddings=[embedding],
                        metadatas=[{"file_name": file, "file_path": file_path}],
                    )
                    print(f"Indexed: {file}")

# Run this once to index all PDFs
index_pdfs(PDF_DIR)



ModuleNotFoundError: No module named 'fitz'

In [ ]:
def search_pdfs(query):
    """Search for relevant PDFs based on a query."""
    query_embedding = embedding_model.encode(query).tolist()
    results = collection.query(query_embeddings=[query_embedding], n_results=3)

    if results["ids"]:
        for i in range(len(results["ids"][0])):
            file_name = results["metadatas"][0][i]["file_name"]
            file_path = results["metadatas"][0][i]["file_path"]
            print(f"Match found in: {file_name} (Path: {file_path})")

        # Query Ollama for a better response
        response = ollama.chat(
            model="gemma:2b",
            messages=[{"role": "user", "content": query}],
        )
        print("\nAI Response:", response["message"]["content"])
    else:
        print("No relevant documents found.")




# Example query
user_query = "What is the main topic of the document?"
search_pdfs(user_query)